# TRICARE PDF → JSON (LlamaParse)

**한 줄 요약**
- PDF > 텍스트 정제 > 단락 분리 > JSON 구조화
- pdf: 파일
- docs: PDF > 파싱 > 페이지별 데이터 리스트

## 1. Import

In [1]:
#!pip install llama-parse

In [2]:
import os
import json
from pathlib import Path
from llama_parse import LlamaParse
from dotenv import load_dotenv

C:\Users\kjw70\AppData\Local\Temp\ipykernel_35184\3502529423.py:4: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


## 2. API Key

In [3]:
load_dotenv()
if not os.getenv("LLAMA_CLOUD_API_KEY"):
    raise ValueError("API KEY 필요")

## 3. Path

In [4]:
PROJECT_ROOT = Path(r"C:\SKN_24\3차 프로젝트\Care-Insure-Bot")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "json_docs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

PROJECT_ROOT: C:\SKN_24\3차 프로젝트\Care-Insure-Bot
DATA_DIR: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\data
OUTPUT_DIR: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json_docs


## 4. Parser

In [5]:
parser = LlamaParse(result_type="markdown")
#print(parser) - api key 노출되서 주석처리 함

## 5. Clean

In [6]:
# 문자열 앞 뒤 공백 제거
# ex. "  TRICARE CONTEXT  " ->"TRICARE CONTEXT"
def clean_text(t):
    return t.strip()

## 6. Split

In [7]:
#텍스트를 단락 단위로 쪼갬
#📌조건 설정: len(p)>30 -> 너무 짧은 문장은 버림(노이즈 제거)
def split_paragraphs(t):
    return [p for p in t.split("\n\n") if len(p)>30]

## 7. PDF→JSON

In [8]:
#LlamaParse가 PDF 읽게 함
#페이지별 문서 리스트 = pdf
def pdf_to_json_from_docs(pdf, docs):
    result = {"file": pdf.name, "pages": []}
    #페이지별 반복
    for i, d in enumerate(docs, 1): # d-> 한 페이지
        text = clean_text(d.text)   # 실제 텍스트
        result["pages"].append({
            "page": i,
            "paragraphs": split_paragraphs(text)# 단락 분리
        })

    return result

---

**중간 점검(체크리스트)**
1. PDF 읽히는 지 확인
2. 원본 텍스트 확인
3. clean_text 확인
4. 단락 분리 확인
5. 전체 JSON 확인

In [9]:
# 디버깅용 테스트 셀
pdf_files = list(DATA_DIR.glob("*.pdf"))
print("PDF 개수:", len(pdf_files))

pdf = pdf_files[0]
print("선택한 파일:", pdf.name)


PDF 개수: 11
선택한 파일: ADDP_Brochure_FINAL_122624_508c.pdf


In [10]:
# 1. parser가 읽은 원본 확인
docs = parser.load_data(str(pdf))
print("페이지 수:", len(docs))

print("\n[첫 페이지 원본 일부]")
print(docs[0].text[:500])

Started parsing the file under job_id 32037b9b-b5af-4047-bcb5-10cc6fc99729
페이지 수: 4

[첫 페이지 원본 일부]

# TRICARE ® Active Duty Dental Program

# The care you need for dental health and readiness

The Active Duty Dental Program provides civilian dental care for active duty service members. The ADDP helps ensure your dental health and deployment readiness. United Concordia Companies, Inc. administers the ADDP benefit. Visit www.addp-ucci.com for more information.

# VERIFYING ELIGIBILITY

Your ADDP eligibility is verified in the Defense Enrollment Eligibility Reporting System. Make sure your infor


In [11]:
# 2. clean_text 적용 확인
cleaned = clean_text(docs[0].text)
print("\n[clean_text 적용 후 일부]")
print(cleaned[:500])


[clean_text 적용 후 일부]
# TRICARE ® Active Duty Dental Program

# The care you need for dental health and readiness

The Active Duty Dental Program provides civilian dental care for active duty service members. The ADDP helps ensure your dental health and deployment readiness. United Concordia Companies, Inc. administers the ADDP benefit. Visit www.addp-ucci.com for more information.

# VERIFYING ELIGIBILITY

Your ADDP eligibility is verified in the Defense Enrollment Eligibility Reporting System. Make sure your inform


In [12]:
# 3. split_paragraphs 적용 확인
paragraphs = split_paragraphs(cleaned)
print("\n[단락 개수]:", len(paragraphs))

for i, p in enumerate(paragraphs[:3], start=1):
    print(f"\n[단락 {i}]")
    print(p[:300])


[단락 개수]: 9

[단락 1]
# TRICARE ® Active Duty Dental Program

[단락 2]
# The care you need for dental health and readiness

[단락 3]
The Active Duty Dental Program provides civilian dental care for active duty service members. The ADDP helps ensure your dental health and deployment readiness. United Concordia Companies, Inc. administers the ADDP benefit. Visit www.addp-ucci.com for more information.


In [13]:
# 4. pdf_to_json 전체 결과 확인
result = pdf_to_json_from_docs(pdf, docs)

print("\n[JSON 결과 구조 확인]")
print("file:", result["file"])
print("pages 수:", len(result["pages"]))

if result["pages"]:
    print("첫 페이지 단락 수:", len(result["pages"][0]["paragraphs"]))


[JSON 결과 구조 확인]
file: ADDP_Brochure_FINAL_122624_508c.pdf
pages 수: 4
첫 페이지 단락 수: 9


---

## 8. Run

In [14]:
pdfs = list(DATA_DIR.glob("*.pdf"))
print("PDF 개수:", len(pdfs))

PDF 개수: 11


In [ ]:
#pdfs = list(DATA_DIR.glob("*.pdf"))
#for p in pdfs:
#    data = pdf_to_json(p)
#    with open(OUTPUT_DIR / (p.stem + ".json"), "w", encoding="utf-8") as f:
#        json.dump(data,f,ensure_ascii=False,indent=2)
# 오류 나서 아래 코드로 수정
#

In [15]:
from llama_parse import LlamaParse
import json

pdfs = list(DATA_DIR.glob("*.pdf"))

print("PDF 개수:", len(pdfs))

for p in pdfs:
    print("\n처리 중:", p.name)

    try:
        parser = LlamaParse(
            result_type="markdown",
            verbose=True
        )

        docs = parser.load_data(str(p))
        data = pdf_to_json_from_docs(p, docs)

        save_path = OUTPUT_DIR / (p.stem + ".json")
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        print("저장 완료:", save_path)
        print("pages 수:", len(data["pages"]))

    except Exception as e:
        print("실패:", p.name)
        print("오류:", e)

PDF 개수: 11

처리 중: ADDP_Brochure_FINAL_122624_508c.pdf
Started parsing the file under job_id 07b94459-f229-466a-89ac-e44ba2b65fc0
저장 완료: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json_docs\ADDP_Brochure_FINAL_122624_508c.json
pages 수: 4

처리 중: Costs_Fees.pdf
Started parsing the file under job_id 29c306cd-0d7e-409e-9788-42f8d95a5c0e
저장 완료: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json_docs\Costs_Fees.json
pages 수: 4

처리 중: Maternity_Br (1).pdf
Started parsing the file under job_id d7aa7ac0-cc89-4c94-8526-5a78712a0ccb
저장 완료: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json_docs\Maternity_Br (1).json
pages 수: 4

처리 중: NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf
Started parsing the file under job_id 076a9f19-f191-4076-bf25-82439a4d97be
저장 완료: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json_docs\NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).json
pages 수: 16

처리 중: Overseas_HB(해외 프로그램 안내서).pdf
Started parsing the file under job_id 01b711d2-55ed-4a00-8990-81a94cb6e677
저장 완료: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outp

In [16]:
# 중간 점검
import json
from pathlib import Path

path = Path(r"C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json_docs\ADDP_Brochure_FINAL_122624_508c.json")

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("file:", data["file"])
print("pages 수:", len(data["pages"]))

file: ADDP_Brochure_FINAL_122624_508c.pdf
pages 수: 4


## 9. Check

In [ ]:
files = list(OUTPUT_DIR.glob("*.json"))
with open(files[0], encoding="utf-8") as f:
    d=json.load(f)
print(d["pages"][0]["paragraphs"][0][:300]) # paragraphs = 단락

# TRICARE ® Active Duty Dental Program


In [20]:
for i, p in enumerate(d["pages"][0]["paragraphs"][:9], start=1):
    print(f"\n[단락 {i}]")
    print(p[:300])


[단락 1]
# TRICARE ® Active Duty Dental Program

[단락 2]
# The care you need for dental health and readiness

[단락 3]
The Active Duty Dental Program provides civilian dental care for active duty service members. The ADDP helps ensure your dental health and deployment readiness. United Concordia Companies, Inc. administers the ADDP benefit. Visit www.addp-ucci.com for more information.

[단락 4]
Your ADDP eligibility is verified in the Defense Enrollment Eligibility Reporting System. Make sure your information in DEERS is up to date. If your eligibility can’t be confirmed, your dental care will be denied.

[단락 5]
Check or update your contact information in DEERS.

[단락 6]
| Online    | <https://milconnect.dmdc.osd.mil>                                                                       |
| --------- | ------------------------------------------------------------------------------------------------------- |
| Phone     | 800-538-9552 or 866-363-2883 (TDD/TTY)        

[단락 7]
CONUS (Continenta

In [ ]:
# 단락 6 
print(d["pages"][0]["paragraphs"][5])

| Online    | <https://milconnect.dmdc.osd.mil>                                                                       |
| --------- | ------------------------------------------------------------------------------------------------------- |
| Phone     | 800-538-9552 or 866-363-2883 (TDD/TTY)                                                                  |
| Fax       | 800-336-4416                                                                                            |
| Mail      | DMDC/DEERS Support Office 400 Gigling Rd. Seaside, CA 93955 United States of America                    |
| In Person | Visit a Uniformed Services ID card office. Find an office near you at <https://idco.dmdc.osd.mil/idco>. |


In [ ]:
# 단락 6
# 문자 수 길이도 체크
p = d["pages"][0]["paragraphs"][5]
print("문자 수:", len(p))
print(p)

문자 수: 719
| Online    | <https://milconnect.dmdc.osd.mil>                                                                       |
| --------- | ------------------------------------------------------------------------------------------------------- |
| Phone     | 800-538-9552 or 866-363-2883 (TDD/TTY)                                                                  |
| Fax       | 800-336-4416                                                                                            |
| Mail      | DMDC/DEERS Support Office 400 Gigling Rd. Seaside, CA 93955 United States of America                    |
| In Person | Visit a Uniformed Services ID card office. Find an office near you at <https://idco.dmdc.osd.mil/idco>. |


In [ ]:
# ADDP_Brochure_FINAL_122624_508c.pdf: 2페이지 확인
print("2페이지 단락 수:", len(d["pages"][1]["paragraphs"]))

2페이지 단락 수: 10


In [ ]:
# Markdown 표
for i, p in enumerate(d["pages"][1]["paragraphs"], start=1):
    print(f"\n[2페이지 단락 {i}]")
    print(p[:500])


[2페이지 단락 1]
The ADDP is available to ADSMs of the:

[2페이지 단락 2]
- U.S. Army
- U.S. Air Force
- U.S. Navy
- U.S. Marine Corps
- U.S. Space Force
- U.S. Coast Guard
- Commissioned Corps of the U.S. Public Health Service
- Commissioned Corps of the National Oceanic and Atmospheric Administration

[2페이지 단락 3]
All others not included in the above eligibility section.

[2페이지 단락 4]
You can find a list of covered dental services in the Benefit Details Document at https://secure.addp-ucci.com/adsm/forms-and-resources. The ADDP doesn’t cover all dental procedures—and you must pay for any non-covered care you choose to receive.

[2페이지 단락 5]
The ADDP doesn’t cover orthodontic services. This includes braces. Check with your service’s policies before getting orthodontics or any other non-covered care. Certain orthodontic appliances could affect your dental readiness.

[2페이지 단락 6]
# Who else is eligible in CONUS locations?

[2페이지 단락 7]
- National Guard and Reserve Members
- National Guard and Reserv

In [ ]:
# 줄 단위 확인
p = d["pages"][1]["paragraphs"][1]

for idx, line in enumerate(p.splitlines(), start=1):
    print(f"{idx:02d}: {line}")

01: - U.S. Army
02: - U.S. Air Force
03: - U.S. Navy
04: - U.S. Marine Corps
05: - U.S. Space Force
06: - U.S. Coast Guard
07: - Commissioned Corps of the U.S. Public Health Service
08: - Commissioned Corps of the National Oceanic and Atmospheric Administration
